In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# ==========================================
# 1) CONFIG
# ==========================================
CSV_FILES = {
    "Alexis": "Alexis_wingate.csv",
    "Antoine": "Antoine_wingate.csv",
    "Jinwei": "Jinwei_wingate.csv",
    "Victor": "Victor_wingate.csv",
}

XLS_FILE = "Physio_Session2_Wingate.xlsx"
SHEET_NAME = "data"

NIRS_CSV = "NIRS_Data_Analysis_2.0_nirs.csv"
PARTICIPANT_CSV = "NIRS_Data_Analysis_2.0_ParticipantData.csv"

WINGATE_DURATION = 30
BASELINE_BEFORE = 15
SLOPE_WINDOW = 10

# ==========================================
# 2) UTILS
# ==========================================
def parse_sync_to_seconds(sync_val, recording_max=None):
    """
    Convertit une synchro Excel en secondes.
    Gère :
    - nombre déjà en secondes
    - texte 'mm:ss'
    - texte 'hh:mm:ss'
    """
    if pd.isna(sync_val):
        raise ValueError("Valeur de synchro vide")

    # déjà numérique
    try:
        x = float(sync_val)
        return x
    except Exception:
        pass

    s = str(sync_val).strip()

    # hh:mm:ss ou mm:ss
    if ":" in s:
        parts = s.split(":")
        parts = [float(p) for p in parts]

        if len(parts) == 2:
            mm, ss = parts
            return mm * 60 + ss
        elif len(parts) == 3:
            hh, mm, ss = parts
            return hh * 3600 + mm * 60 + ss

    raise ValueError(f"Format synchro non reconnu: {sync_val}")


def load_trainred_csv(file_path):
    """
    Lit un CSV Train.Red et renvoie Time, SmO2.
    """
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    header_idx = None
    for i, line in enumerate(lines):
        if "Timestamp (seconds passed)" in line and "SmO2" in line:
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(f"Impossible de trouver l'en-tête SmO2 dans {file_path}")

    df = pd.read_csv(file_path, skiprows=header_idx)
    df.columns = [str(c).strip() for c in df.columns]

    df = df[["Timestamp (seconds passed)", "SmO2"]].copy()
    df.columns = ["Time", "SmO2"]

    df["Time"] = pd.to_numeric(df["Time"], errors="coerce")
    df["SmO2"] = pd.to_numeric(df["SmO2"], errors="coerce")
    df = df.dropna(subset=["Time", "SmO2"]).sort_values("Time").reset_index(drop=True)

    return df


def extract_wingate_window(sm_df, start_time_s, duration_s=30, baseline_before_s=15):
    """
    Extrait baseline + fenêtre Wingate.
    Retourne des features SmO2.
    """
    sm = sm_df.copy()

    baseline_df = sm[(sm["Time"] >= start_time_s - baseline_before_s) &
                     (sm["Time"] < start_time_s)]

    if baseline_df.empty:
        baseline_df = sm[sm["Time"] < start_time_s]

    if baseline_df.empty:
        raise ValueError(f"Pas de baseline possible pour ce sujet (start={start_time_s})")

    sm0 = float(baseline_df["SmO2"].mean())

    win = sm[(sm["Time"] >= start_time_s) &
             (sm["Time"] <= start_time_s + duration_s)].copy()

    if len(win) < 5:
        raise ValueError("Fenêtre Wingate trop courte")

    win["t_rel"] = win["Time"] - start_time_s
    win["dSmO2"] = (sm0 - win["SmO2"]).clip(lower=0)

    dt = float(np.mean(np.diff(win["Time"])))
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("dt invalide dans la fenêtre Wingate")

    auc_30 = float(np.sum(win["dSmO2"]) * dt)
    min_smo2 = float(win["SmO2"].min())
    dmax = float(win["dSmO2"].max())

    early = win[win["t_rel"] <= SLOPE_WINDOW].copy()
    if len(early) >= 3:
        slope_0_10 = float(np.polyfit(early["t_rel"], early["SmO2"], 1)[0])
    else:
        slope_0_10 = np.nan

    return {
        "sm_window": win,
        "SmO2_baseline": sm0,
        "AUC_30s": auc_30,
        "SmO2_min": min_smo2,
        "dSmO2_max": dmax,
        "slope_0_10": slope_0_10
    }


def summarize_lactate_subject(lac_subject):
    """
    Résumé lactate d'un sujet.
    baseline = Time == -1
    effort = Time == 0
    post = Time > 0
    """
    lac_subject = lac_subject.sort_values("Time").copy()

    base_row = lac_subject[lac_subject["Time"] == -1]
    if base_row.empty:
        raise ValueError("Pas de lactate baseline (Time == -1)")
    la_baseline = float(base_row["[La]"].iloc[0])

    effort_row = lac_subject[lac_subject["Time"] == 0]
    la_effort0 = float(effort_row["[La]"].iloc[0]) if not effort_row.empty else np.nan

    post = lac_subject[lac_subject["Time"] > 0]
    if post.empty:
        raise ValueError("Pas de lactates post-effort")

    idx_peak = post["[La]"].idxmax()
    la_peak = float(post.loc[idx_peak, "[La]"])
    t_peak = float(post.loc[idx_peak, "Time"])

    delta_peak = la_peak - la_baseline

    return {
        "La_baseline": la_baseline,
        "La_effort0": la_effort0,
        "La_peak": la_peak,
        "Time_peak_min": t_peak,
        "Delta_La_peak": delta_peak
    }

# ==========================================
# 3) TES DONNÉES WINGATE
# ==========================================
lac_all = pd.read_excel(XLS_FILE, sheet_name=SHEET_NAME)
lac_all.columns = [str(c).strip() for c in lac_all.columns]

required_cols = ["Name", "Time", "[La]", "Synchro SmO2"]
for c in required_cols:
    if c not in lac_all.columns:
        raise ValueError(f"Colonne manquante dans l'Excel: {c}")

lac_all["Time"] = pd.to_numeric(lac_all["Time"], errors="coerce")
lac_all["[La]"] = pd.to_numeric(lac_all["[La]"], errors="coerce")

rows = []
windows = {}

for subject, csv_file in CSV_FILES.items():
    print(f"\n--- {subject} ---")

    sm = load_trainred_csv(csv_file)
    recording_max = float(sm["Time"].max())

    lac_sub = lac_all[lac_all["Name"].str.lower() == subject.lower()].copy()
    if lac_sub.empty:
        print(f"{subject}: pas trouvé dans l'Excel")
        continue

    sync_val = lac_sub["Synchro SmO2"].dropna().iloc[0]
    start_time_s = parse_sync_to_seconds(sync_val, recording_max)

    sm_feat = extract_wingate_window(
        sm_df=sm,
        start_time_s=start_time_s,
        duration_s=WINGATE_DURATION,
        baseline_before_s=BASELINE_BEFORE
    )

    la_feat = summarize_lactate_subject(lac_sub)

    row = {
        "id": subject,
        "start_time_s": start_time_s,
        **{k: v for k, v in sm_feat.items() if k != "sm_window"},
        **la_feat
    }
    rows.append(row)
    windows[subject] = sm_feat["sm_window"]

feat_df = pd.DataFrame(rows)

print("\n===== FEATURES WINGATE =====")
print(feat_df)

# ==========================================
# 4) CHARGER LES 2 CSV DE L’ARTICLE
# ==========================================
nirs_raw = pd.read_csv(NIRS_CSV, sep=";")
part_raw = pd.read_csv(PARTICIPANT_CSV, sep=";")

# Le fichier NIRS a sa vraie ligne d’en-tête dans la 1ère ligne de données
nirs_raw.columns = nirs_raw.iloc[0]
nirs_raw = nirs_raw.iloc[1:].reset_index(drop=True)

# Renommer les colonnes 10s / 15s / 30s à la main
nirs_cols = [
    "Participant",
    "Skinfold",
    "10s_SmO2_lowest",
    "10s_SmO2_max",
    "10s_dSmO2",
    "10s_t_half",
    "10s_MRT_tau",
    "10s_time_to_desat",
    "10s_linear_desat_first10",
    "10s_max_linear_desat",
    "10s_linear_resat_first18",
    "10s_max_linear_resat",
    "10s_r2_recovery",
    "15s_SmO2_lowest",
    "15s_SmO2_max",
    "15s_dSmO2",
    "15s_t_half",
    "15s_MRT_tau",
    "15s_time_to_desat",
    "15s_linear_desat_first10",
    "15s_max_linear_desat",
    "15s_linear_resat_first18",
    "15s_max_linear_resat",
    "15s_r2_recovery",
    "30s_SmO2_lowest",
    "30s_SmO2_max",
    "30s_dSmO2",
    "30s_t_half",
    "30s_MRT_tau",
    "30s_time_to_desat",
    "30s_linear_desat_first10",
    "30s_max_linear_desat",
    "30s_linear_resat_first18",
    "30s_max_linear_resat",
    "30s_r2_recovery",
    "30s_desat_t_half",
    "30s_desat_tau_MRT",
    "30s_r2_desat"
]
nirs_raw.columns = nirs_cols

# Nettoyage participant data
part = part_raw.copy()
part.columns = [str(c).strip() for c in part.columns]

# Numeric conversion
for col in nirs_raw.columns:
    nirs_raw[col] = pd.to_numeric(nirs_raw[col], errors="coerce")

for col in part.columns:
    part[col] = pd.to_numeric(part[col], errors="coerce")

# ==========================================
# 5) CONSTRUIRE UN TABLEAU 30s POUR vLamax
# ==========================================
article_30 = pd.DataFrame({
    "Participant": nirs_raw["Participant"],
    "SmO2_min": nirs_raw["30s_SmO2_lowest"],
    "SmO2_max": nirs_raw["30s_SmO2_max"],
    "dSmO2": nirs_raw["30s_dSmO2"],
    "linear_desat_first10": nirs_raw["30s_linear_desat_first10"],
    "max_linear_desat": nirs_raw["30s_max_linear_desat"],
    "linear_resat_first18": nirs_raw["30s_linear_resat_first18"],
    "max_linear_resat": nirs_raw["30s_max_linear_resat"],
    "time_to_desat": nirs_raw["30s_time_to_desat"],
    "vLamax_30s": part["30s. V̇Lamax"]
})

article_30 = article_30.dropna().reset_index(drop=True)

print("\n===== ARTICLE DATASET 30s =====")
print(article_30.head())
print(f"N article_30 = {len(article_30)}")

# ==========================================
# 6) MODELE 1 : SmO2 -> vLamax
# ==========================================
# On prend des variables compatibles avec ton Wingate :
# - SmO2_min
# - dSmO2
# - linear_desat_first10
#
# Note :
# linear_desat_first10 dans l’article est positive (vitesse de baisse),
# alors que ton slope_0_10 est négatif car il vient d’une régression sur SmO2 brute.
# Donc on utilisera : desat_rate_equiv = -slope_0_10

X_vla = article_30[["SmO2_min", "dSmO2", "linear_desat_first10"]].to_numpy(float)
y_vla = article_30["vLamax_30s"].to_numpy(float)

model_vla = LinearRegression()
model_vla.fit(X_vla, y_vla)

article_30["vLamax_pred"] = model_vla.predict(X_vla)

r2_vla = r2_score(y_vla, article_30["vLamax_pred"])
rmse_vla = np.sqrt(mean_squared_error(y_vla, article_30["vLamax_pred"]))

print("\n===== MODELE 1 : SmO2 -> vLamax =====")
print("Intercept :", model_vla.intercept_)
print("Coef      :", dict(zip(["SmO2_min", "dSmO2", "linear_desat_first10"], model_vla.coef_)))
print(f"R²        = {r2_vla:.3f}")
print(f"RMSE      = {rmse_vla:.3f}")

# ==========================================
# 7) APPLIQUER LE MODELE vLamax À TES 4 WINGATE
# ==========================================
feat_df["linear_desat_equiv"] = -feat_df["slope_0_10"]

X_wing_vla = feat_df[["SmO2_min", "dSmO2_max", "linear_desat_equiv"]].copy()
X_wing_vla.columns = ["SmO2_min", "dSmO2", "linear_desat_first10"]

feat_df["vLamax_pred"] = model_vla.predict(X_wing_vla.to_numpy(float))

print("\n===== vLamax PRÉDIT POUR TES 4 SUJETS =====")
print(feat_df[["id", "SmO2_min", "dSmO2_max", "slope_0_10", "vLamax_pred"]])

# ==========================================
# 8) MODELE 2 : vLamax -> lactate pic
# ==========================================
X_la_vla = feat_df[["vLamax_pred"]].to_numpy(float)
y_la = feat_df["Delta_La_peak"].to_numpy(float)

model_la_from_vla = LinearRegression()
model_la_from_vla.fit(X_la_vla, y_la)

feat_df["DeltaLa_pred_from_vla"] = model_la_from_vla.predict(X_la_vla)

r2_la_vla = r2_score(y_la, feat_df["DeltaLa_pred_from_vla"])
rmse_la_vla = np.sqrt(mean_squared_error(y_la, feat_df["DeltaLa_pred_from_vla"]))

print("\n===== MODELE 2 : vLamax -> Delta lactate pic =====")
print("Intercept :", model_la_from_vla.intercept_)
print("Coef      :", {"vLamax_pred": model_la_from_vla.coef_[0]})
print(f"R²        = {r2_la_vla:.3f}")
print(f"RMSE      = {rmse_la_vla:.3f}")

# ==========================================
# 9) MODELE DIRECT (ton baseline actuel)
# ==========================================
X_direct = feat_df[["AUC_30s"]].to_numpy(float)
model_direct = LinearRegression()
model_direct.fit(X_direct, y_la)

feat_df["DeltaLa_pred_direct"] = model_direct.predict(X_direct)

r2_direct = r2_score(y_la, feat_df["DeltaLa_pred_direct"])
rmse_direct = np.sqrt(mean_squared_error(y_la, feat_df["DeltaLa_pred_direct"]))

print("\n===== MODELE DIRECT : AUC_30s -> Delta lactate pic =====")
print("Intercept :", model_direct.intercept_)
print("Coef      :", {"AUC_30s": model_direct.coef_[0]})
print(f"R²        = {r2_direct:.3f}")
print(f"RMSE      = {rmse_direct:.3f}")

# ==========================================
# 10) COMPARAISON DES 2 MODELES
# ==========================================
comparison = feat_df[[
    "id",
    "Delta_La_peak",
    "DeltaLa_pred_direct",
    "vLamax_pred",
    "DeltaLa_pred_from_vla"
]].copy()

print("\n===== COMPARAISON DES PRÉDICTIONS =====")
print(comparison)

# ==========================================
# 11) PLOTS
# ==========================================
# 11A - Article : vrai vLamax vs prédit
plt.figure(figsize=(6, 5))
plt.scatter(article_30["vLamax_30s"], article_30["vLamax_pred"])
mn = min(article_30["vLamax_30s"].min(), article_30["vLamax_pred"].min())
mx = max(article_30["vLamax_30s"].max(), article_30["vLamax_pred"].max())
plt.plot([mn, mx], [mn, mx], "--")
plt.xlabel("vLamax mesuré (30s)")
plt.ylabel("vLamax prédit")
plt.title("Modèle 1 — SmO2 -> vLamax")
plt.grid(True)
plt.show()

# 11B - Direct model
plt.figure(figsize=(6, 4))
plt.scatter(feat_df["AUC_30s"], feat_df["Delta_La_peak"], label="Mesuré")
xx = np.linspace(feat_df["AUC_30s"].min(), feat_df["AUC_30s"].max(), 100)
yy = model_direct.predict(xx.reshape(-1, 1))
plt.plot(xx, yy, label="Modèle direct")
for _, r in feat_df.iterrows():
    plt.annotate(r["id"], (r["AUC_30s"], r["Delta_La_peak"]), xytext=(5, 3), textcoords="offset points")
plt.xlabel("AUC SmO2 sur 30 s")
plt.ylabel("Delta lactate pic (mmol/L)")
plt.title("Modèle direct — AUC -> lactate")
plt.grid(True)
plt.legend()
plt.show()

# 11C - Physiological model
plt.figure(figsize=(6, 4))
plt.scatter(feat_df["vLamax_pred"], feat_df["Delta_La_peak"], label="Mesuré")
xx = np.linspace(feat_df["vLamax_pred"].min(), feat_df["vLamax_pred"].max(), 100)
yy = model_la_from_vla.predict(xx.reshape(-1, 1))
plt.plot(xx, yy, label="Modèle physiologique")
for _, r in feat_df.iterrows():
    plt.annotate(r["id"], (r["vLamax_pred"], r["Delta_La_peak"]), xytext=(5, 3), textcoords="offset points")
plt.xlabel("vLamax prédit")
plt.ylabel("Delta lactate pic (mmol/L)")
plt.title("Modèle physiologique — vLamax -> lactate")
plt.grid(True)
plt.legend()
plt.show()

# 11D - comparaison sujet par sujet
plt.figure(figsize=(8, 4))
x = np.arange(len(feat_df))
width = 0.25

plt.bar(x - width, feat_df["Delta_La_peak"], width=width, label="Mesuré")
plt.bar(x, feat_df["DeltaLa_pred_direct"], width=width, label="Prédit direct")
plt.bar(x + width, feat_df["DeltaLa_pred_from_vla"], width=width, label="Prédit via vLamax")

plt.xticks(x, feat_df["id"])
plt.ylabel("Delta lactate pic")
plt.title("Comparaison des modèles")
plt.legend()
plt.grid(True, axis="y")
plt.show()